# Langchain embedding
1. OpenAI
2. Ollama - not practicing it since mac os - ventura - 13.4 dont support it.
3. Hugging face 

In [ ]:
# load environment variables, install python-dotenv
import os
import dotenv
from dotenv import load_dotenv
#load environemt file
load_dotenv()

os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")
os.environ["LANGCHAIN_TRACKING_V2"] = "true"
os.environ["open_ai_key"] = os.getenv("OPENAI_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")


# load and split data

In [ ]:
# get data and split it
from langchain_text_splitters import HTMLHeaderTextSplitter

headers = [
    ("h1","header1"),
    ("h2","header2"),
    ("h3","header3"),
]
splitter = HTMLHeaderTextSplitter(headers_to_split_on=headers)
htmlSplit = splitter.split_text_from_file("example.html")
len(htmlSplit)
htmlText = [content.page_content for content in htmlSplit]

# OpenAI

Its paid. 
1. Get api key from open ai website.
2. get name of embedding model

embed_documents() => List[str] => List[List[float]] => Embedding your chunks once, at ingestion time
embed_query() => str (single string)	=> List[float] (single vector) => Embedding the user's search question, at retrieval time

In [ ]:
from langchain_openai import OpenAIEmbeddings


embed = OpenAIEmbeddings(model="text-embedding-3-small")
htmlEmbed = embed.embed_documents(htmlText)
print(type(htmlEmbed),len(htmlEmbed))
print(htmlEmbed)


In [ ]:
print(len(htmlEmbed)) #number of chunks
print(len(htmlEmbed[0]))# dimension of single embedding vector

tiktoken (OpenAI's tokenizer, published as a library)<br>
    ↓<br>
    ├── used internally by OpenAI's API servers to tokenize/bill your requests<br>
    └── used by LangChain's TokenTextSplitter (your local code) to count tokens while <br>chunking

In [ ]:
# how it works internally for cost - get number of tokens
import tiktoken

encode = tiktoken.encoding_for_model(model_name="text-embedding-3-small")
total_tokens = sum(len(encode.encode(t)) for t in htmlText)
print("Number of tokens :",total_tokens)

# Hugging face

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embed = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
htmlEmbed = embed.embed_documents(htmlText)
print(type(htmlEmbed),len(htmlEmbed))
print(len(htmlEmbed[0]))

Coding <br>

Problem 1
<br>
Generate embeddings for:
<br>
"I love machine learning"<br>
"I enjoy studying AI"<br>
"I like cooking"<br>
<br>
Calculate cosine similarity.<br>


In [ ]:
#problem 1 

from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

embedding = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
txt1 = embedding.embed_query("I love machine learning")
txt2 = embedding.embed_query("I enjoy studying AI")
txt3 = embedding.embed_query("I like cooking")

def cosine_similarity(embed1, embed2):
    return np.dot(embed1,embed2)/ (np.linalg.norm(embed1) * np.linalg.norm(embed2))

t12 = cosine_similarity(txt1,txt2)
t23 = cosine_similarity(txt2,txt3)
t13 = cosine_similarity(txt1,txt3)

print(f"Cosinse simpilaries betwee txt1, 2 and 3: {t12},{t23},{t13}")


<br>
Problem 2<br>
<br>
Write cosine similarity manually using NumPy.<br>

In [ ]:
#Problem 2

def cosine_similarity(embed1, embed2):
    return np.dot(embed1,embed2)/ (np.linalg.norm(embed1) * np.linalg.norm(embed2))


<br>
Problem 3 ⭐<br>
<br>
Build semantic search without LangChain:<br>

documents<br>
   ↓<br>
embeddings<br>
   ↓<br>
query embedding<br>
   ↓<br>
cosine similarity<br>
   ↓<br>
top 3 documents<br>



In [ ]:
#Problem 3 

from sentence_transformers import SentenceTransformer
import numpy as np

documents = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks.",
    "Python is used for data science.",
    "Cooking requires different ingredients.",
    "Football is a popular sport."
]

model = SentenceTransformer(model_name_or_path="all-MiniLM-L6-v2")
docEmbeddings = model.encode(documents)

query = "What are neural networks?"
queryEmbed = model.encode(query)

print(docEmbeddings.shape, queryEmbed.shape, queryEmbed[0].shape)

scores = []
for doc in docEmbeddings:
    print(doc.shape)
    score = cosine_similarity(queryEmbed,doc)
    scores.append(score)

print(scores)
scores = np.array(scores)
print(scores, scores.shape)

top_indices = scores.argsort()[-3:][::-1]
print(top_indices)

for i in top_indices:
    print(documents[i], scores[i])


Problem 4 ⭐<br>

Implement the same search using LangChain embeddings.<br>



In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.utils.math import cosine_similarity_top_k,cosine_similarity

huggingFaceEmbed = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
docEmbeddings = huggingFaceEmbed.embed_documents(documents)
queryEmbed = huggingFaceEmbed.embed_query(query)

print(len(queryEmbed),len(docEmbeddings[0]))

queryEmbed = np.array(queryEmbed).reshape((1,-1))
print(queryEmbed.shape)

scores = []
for docs in docEmbeddings:
    Y = np.array(docs).reshape(1,-1)
    score = cosine_similarity(queryEmbed,Y)
    scores.append(score)

scores = np.array(scores).reshape(-1)
print(scores)
top_indices  = scores.argsort()[-3:][::-1]
print(top_indices)

print("***** USing cosine similarity top k")
docEmbeddings = np.array(docEmbeddings)
print(docEmbeddings.shape)
index, scores = cosine_similarity_top_k(queryEmbed,docEmbeddings,top_k=3)
print(index,scores)




Problem 5 Advanced<br>

Compare two embedding models on the same 20 queries and determine which retrieves more semantically relevant documents.<br>

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np



openai_model = OpenAIEmbeddings(model = "text-embedding-3-small")
huggingface_model = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")
models = {
    "model1_openAI":openai_model,
    "model2_huggingFace":huggingface_model
}

documents = [
    {
        "id": "D1",
        "text": "Python lists are ordered, mutable collections that can contain elements of different data types. They support indexing, slicing, appending, removing, and sorting elements."
    },
    {
        "id": "D2",
        "text": "Python dictionaries store data as key-value pairs. They provide fast lookup using keys and are commonly used when data needs to be associated with meaningful identifiers."
    },
    {
        "id": "D3",
        "text": "Python sets are unordered collections of unique elements. They are useful for removing duplicates and performing union, intersection, and difference operations."
    },
    {
        "id": "D4",
        "text": "Python uses try, except, else, and finally blocks to handle runtime errors. Exception handling allows programs to recover gracefully from unexpected situations."
    },
    {
        "id": "D5",
        "text": "NumPy arrays provide efficient multidimensional numerical data structures. They support vectorized operations, broadcasting, reshaping, and mathematical computations."
    },
    {
        "id": "D6",
        "text": "NumPy broadcasting allows arithmetic operations between arrays with compatible but different shapes without explicitly copying data."
    },
    {
        "id": "D7",
        "text": "A Pandas DataFrame is a two-dimensional labeled data structure used for manipulating and analyzing tabular data."
    },
    {
        "id": "D8",
        "text": "Pandas provides functions such as isna(), fillna(), and dropna() for detecting, replacing, and removing missing values in datasets."
    },
    {
        "id": "D9",
        "text": "The Pandas groupby operation divides data into groups based on one or more columns and allows aggregation such as sum, mean, count, and maximum."
    },
    {
        "id": "D10",
        "text": "Machine learning datasets are commonly divided into training and testing subsets. The model learns from training data and is evaluated on unseen testing data."
    },
    {
        "id": "D11",
        "text": "Cross-validation repeatedly divides available data into training and validation portions to obtain a more reliable estimate of model performance."
    },
    {
        "id": "D12",
        "text": "Overfitting occurs when a machine learning model learns the training data too closely, including noise, and performs poorly on unseen data."
    },
    {
        "id": "D13",
        "text": "Regularization techniques such as L1 and L2 penalties discourage overly complex models and can reduce overfitting."
    },
    {
        "id": "D14",
        "text": "Gradient descent is an optimization algorithm that iteratively updates model parameters in the direction that reduces the loss function."
    },
    {
        "id": "D15",
        "text": "Embeddings represent text, images, or other data as dense numerical vectors. Semantically similar items tend to have similar vector representations."
    },
    {
        "id": "D16",
        "text": "Cosine similarity measures the similarity between two vectors based on the angle between them. It is frequently used to compare embedding vectors."
    },
    {
        "id": "D17",
        "text": "Vector databases store and index vector representations to enable efficient similarity search over large collections of embeddings."
    },
    {
        "id": "D18",
        "text": "Retrieval-Augmented Generation combines information retrieval with language generation. Relevant documents are retrieved and provided to an LLM as context for generating an answer."
    },
    {
        "id": "D19",
        "text": "Text chunking divides large documents into smaller pieces before embedding. Appropriate chunk sizes help retrieval systems find relevant information while preserving useful context."
    },
    {
        "id": "D20",
        "text": "Prompt injection is an attack in which malicious instructions are inserted into input data to manipulate an LLM into ignoring its intended instructions or revealing sensitive information."
    }
]

queries = [
    {
        "id": "Q1",
        "text": "How can I keep only distinct values in a collection?"
    },
    {
        "id": "Q2",
        "text": "What Python structure lets me retrieve information using a named key?"
    },
    {
        "id": "Q3",
        "text": "How do I safely deal with errors that happen while my Python program is running?"
    },
    {
        "id": "Q4",
        "text": "What data structure should I use when I need an ordered collection that I can modify?"
    },
    {
        "id": "Q5",
        "text": "How can I perform mathematical operations efficiently on large numerical arrays?"
    },
    {
        "id": "Q6",
        "text": "How can arrays with different dimensions work together during arithmetic in NumPy?"
    },
    {
        "id": "Q7",
        "text": "What Python library structure is best suited for working with rows and columns of data?"
    },
    {
        "id": "Q8",
        "text": "How can I replace empty or unavailable values in a tabular dataset?"
    },
    {
        "id": "Q9",
        "text": "How can I calculate the average salary for each department using Pandas?"
    },
    {
        "id": "Q10",
        "text": "Why should I keep some data aside instead of using everything to train my model?"
    },
    {
        "id": "Q11",
        "text": "How can I get a more dependable estimate of how well my ML model generalizes?"
    },
    {
        "id": "Q12",
        "text": "Why does a model perform extremely well on training data but poorly on new data?"
    },
    {
        "id": "Q13",
        "text": "What techniques can discourage a machine learning model from becoming too complicated?"
    },
    {
        "id": "Q14",
        "text": "How does an ML algorithm iteratively find parameters that minimize its error?"
    },
    {
        "id": "Q15",
        "text": "How can words or sentences be converted into numerical representations that preserve meaning?"
    },
    {
        "id": "Q16",
        "text": "How can I determine whether two embedding vectors point in similar directions?"
    },
    {
        "id": "Q17",
        "text": "What technology can efficiently search a large collection of vectors for similar ones?"
    },
    {
        "id": "Q18",
        "text": "How can an LLM use information retrieved from my own documents to answer questions?"
    },
    {
        "id": "Q19",
        "text": "Why do RAG systems divide large documents into smaller pieces before creating embeddings?"
    },
    {
        "id": "Q20",
        "text": "What type of attack attempts to manipulate an LLM through malicious instructions in its input?"
    }
]

relevant_docs = {
    "Q1": ["D3"],
    "Q2": ["D2"],
    "Q3": ["D4"],
    "Q4": ["D1"],
    "Q5": ["D5"],
    "Q6": ["D6"],
    "Q7": ["D7"],
    "Q8": ["D8"],
    "Q9": ["D9"],
    "Q10": ["D10"],
    "Q11": ["D11"],
    "Q12": ["D12"],
    "Q13": ["D13"],
    "Q14": ["D14"],
    "Q15": ["D15"],
    "Q16": ["D16"],
    "Q17": ["D17"],
    "Q18": ["D18"],
    "Q19": ["D19"],
    "Q20": ["D20"]
}

In [ ]:
for i,query  in enumerate(queries):
    print(query.keys())

In [ ]:

def cosine_similarity(embed1, embed2):
    return np.dot(embed1,embed2)/ (np.linalg.norm(embed1) * np.linalg.norm(embed2))

results = {}
for modelName, model in models.items():
    print(modelName)
    docEmbeddings = model.embed_documents([doc["text"] for doc in documents])
    queryEmbeddings = model.embed_documents([query["text"] for query in queries])
    print(len(docEmbeddings),len(queryEmbeddings))
    #calcualte cosine similarity
    query_scores = []
    is_hitModel = []
    for i,query  in enumerate(queries):
        scores = []
        queryScDict = {}
        for j, doc in enumerate(documents):
            score = cosine_similarity(queryEmbeddings[i],docEmbeddings[j])
            # print(scores)
            scores.append(score)
        top_indices = np.argsort(np.array(scores))[-3:].tolist()
        dict_ids = [documents[k]["id"] for k in top_indices]

        queryScDict.update({query.get('id'):dict_ids})
        query_scores.append(queryScDict)

        is_hit = any(doc_id in relevant_docs[query.get('id')] for doc_id in dict_ids)
        is_hitModel.append(is_hit)
    results[modelName] = query_scores
    hit_3 = sum(is_hitModel)/len(queries)
    print(f"For model , {modelName}: {hit_3}")
